<a href="https://cognitiveclass.ai"><img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DL0101EN-SkillsNetwork/images/IDSN-logo.png" width="400"> </a>

# Convolutional Neural Networks with Keras

Estimated time needed **30** mins


In this lab, we will learn how to use the Keras library to build convolutional neural networks. We will also use the popular MNIST dataset and we will compare our results to using a conventional neural network.


## Objectives for this Notebook    
* How to use the Keras library to build convolutional neural networks
* Convolutional neural network with one set of convolutional and pooling layers
* Convolutional neural network with two sets of convolutional and pooling layers



## Table of Contents

<div class="alert alert-block alert-info" style="margin-top: 20px">

<font size = 3>
      
1. <a href="#Import-Keras-and-Packages">Import Keras and Packages</a>   
2. <a href="#Convolutional-Neural-Network-with-One-Set-of-Convolutional-and-Pooling-Layers">Convolutional Neural Network with One Set of Convolutional and Pooling Layers</a>  
3. <a href="#Convolutional-Neural-Network-with-Two-Sets-of-Convolutional-and-Pooling-Layers">Convolutional Neural Network with Two Sets of Convolutional and Pooling Layers</a>  

</font>
</div>


### Install the necessary libraries


Let's start by installing the keras libraries and the packages that we would need to build a neural network.


In [1]:
# All Libraries required for this lab are listed below. The libraries need to be installed on Skills Network Labs. 
# If you run this notebook on a different environment, e.g. your desktop, you may want to install these.
!pip install numpy==2.0.2
!pip install pandas==2.2.2
!pip install tensorflow_cpu==2.18.0
!pip install matplotlib==3.9.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 79.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 79.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.3/230.3 MB 28.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 95.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 49.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 85.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

#### Suppress the tensorflow warning messages
We use the following code to  suppress the warning messages due to use of CPU architechture for tensoflow.
You may want to **comment out** these lines if you are using the GPU architechture


In [2]:
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

## Import Keras and Packages


In [3]:
import keras
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import Input
from keras.utils import to_categorical

When working with convolutional neural networks in particular, we will need additional packages.


In [4]:
from keras.layers import Conv2D # to add convolutional layers
from keras.layers import MaxPooling2D # to add pooling layers
from keras.layers import Flatten # to flatten data for fully connected layers

## Convolutional Neural Network with One Set of Convolutional and Pooling Layers


In [5]:
# import data
from keras.datasets import mnist

# load data
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# reshape to be [samples][pixels][width][height]
X_train = X_train.reshape(X_train.shape[0], 28, 28, 1).astype('float32')
X_test = X_test.reshape(X_test.shape[0], 28, 28, 1).astype('float32')

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Let's normalize the pixel values to be between 0 and 1


In [6]:
X_train = X_train / 255 # normalize training data
X_test = X_test / 255 # normalize test data

Next, let's convert the target variable into binary categories


In [7]:
y_train = to_categorical(y_train)
y_test = to_categorical(y_test)

num_classes = y_test.shape[1] # number of categories

Next, let's define a function that creates our model. Let's start with one set of convolutional and pooling layers.


In [8]:
def convolutional_model():
    
    # create model
    model = Sequential()
    model.add(Input(shape=(28, 28, 1)))
    model.add(Conv2D(16, (5, 5), strides=(1, 1), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2), strides=(2, 2)))
    
    model.add(Flatten())
    model.add(Dense(100, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))
    
    # compile model
    model.compile(optimizer='adam', loss='categorical_crossentropy',  metrics=['accuracy'])
    return model

Finally, let's call the function to create the model, and then let's train it and evaluate it.


In [9]:
# build the model
model = convolutional_model()

# fit the model
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=10, batch_size=200, verbose=2)

# evaluate the model
scores = model.evaluate(X_test, y_test, verbose=0)
print("Accuracy: {} \n Error: {}".format(scores[1], 100-scores[1]*100))

Epoch 1/10
300/300 - 43s - 145ms/step - accuracy: 0.9234 - loss: 0.2725 - val_accuracy: 0.9706 - val_loss: 0.0956
Epoch 2/10
300/300 - 34s - 112ms/step - accuracy: 0.9757 - loss: 0.0824 - val_accuracy: 0.9798 - val_loss: 0.0620
Epoch 3/10
300/300 - 33s - 111ms/step - accuracy: 0.9834 - loss: 0.0563 - val_accuracy: 0.9846 - val_loss: 0.0463
Epoch 4/10
300/300 - 34s - 113ms/step - accuracy: 0.9871 - loss: 0.0430 - val_accuracy: 0.9838 - val_loss: 0.0494
Epoch 5/10
300/300 - 33s - 110ms/step - accuracy: 0.9892 - loss: 0.0357 - val_accuracy: 0.9864 - val_loss: 0.0394
Epoch 6/10
300/300 - 33s - 112ms/step - accuracy: 0.9912 - loss: 0.0287 - val_accuracy: 0.9848 - val_loss: 0.0477
Epoch 7/10
300/300 - 34s - 114ms/step - accuracy: 0.9925 - loss: 0.0249 - val_accuracy: 0.9891 - val_loss: 0.0339
Epoch 8/10
300/300 - 34s - 114ms/step - accuracy: 0.9938 - loss: 0.0203 - val_accuracy: 0.9883 - val_loss: 0.0352
Epoch 9/10
300/300 - 33s - 110ms/step - accuracy: 0.9953 - loss: 0.0162 - val_accuracy: 

------------------------------------------


## Convolutional Neural Network with Two Sets of Convolutional and Pooling Layers


Let's redefine our convolutional model so that it has two convolutional and pooling layers instead of just one layer of each.


In [10]:
def convolutional_model():
    
    # create model
    model = Sequential()
    model.add(Input(shape=(28, 28, 1)))
    model.add(Conv2D(16, (5, 5), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2), strides=(2, 2)))
    
    model.add(Conv2D(8, (2, 2), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2), strides=(2, 2)))
    
    model.add(Flatten())
    model.add(Dense(100, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))
    
    # Compile model
    model.compile(optimizer='adam', loss='categorical_crossentropy',  metrics=['accuracy'])
    return model

Now, let's call the function to create our new convolutional neural network, and then let's train it and evaluate it.


In [12]:
# build the model
model = convolutional_model()

# fit the model
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=10, batch_size=200, verbose=2)

# evaluate the model
scores = model.evaluate(X_test, y_test, verbose=0)
print("Accuracy: {} \n Error: {}".format(scores[1], 100-scores[1]*100))

Epoch 1/10
300/300 - 37s - 124ms/step - accuracy: 0.8578 - loss: 0.5023 - val_accuracy: 0.9569 - val_loss: 0.1422
Epoch 2/10
300/300 - 35s - 118ms/step - accuracy: 0.9656 - loss: 0.1163 - val_accuracy: 0.9724 - val_loss: 0.0865
Epoch 3/10
300/300 - 35s - 118ms/step - accuracy: 0.9750 - loss: 0.0838 - val_accuracy: 0.9818 - val_loss: 0.0577
Epoch 4/10
300/300 - 35s - 118ms/step - accuracy: 0.9797 - loss: 0.0683 - val_accuracy: 0.9793 - val_loss: 0.0627
Epoch 5/10
300/300 - 37s - 122ms/step - accuracy: 0.9823 - loss: 0.0594 - val_accuracy: 0.9836 - val_loss: 0.0505
Epoch 6/10
300/300 - 37s - 122ms/step - accuracy: 0.9842 - loss: 0.0528 - val_accuracy: 0.9852 - val_loss: 0.0466
Epoch 7/10
300/300 - 37s - 123ms/step - accuracy: 0.9858 - loss: 0.0474 - val_accuracy: 0.9864 - val_loss: 0.0438
Epoch 8/10
300/300 - 37s - 125ms/step - accuracy: 0.9873 - loss: 0.0431 - val_accuracy: 0.9878 - val_loss: 0.0368
Epoch 9/10
300/300 - 37s - 124ms/step - accuracy: 0.9880 - loss: 0.0401 - val_accuracy: 

<h3>Practice Exercise 1</h3>


Let's see how batch size affects the time required and accuracy of the model training. 
For this, you can try to change batch_size to 1024 and check it's effect on accuracy


In [13]:
# Write your answer here

# build the model
model = convolutional_model()

# fit the model
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=10, batch_size=1024, verbose=2)

# evaluate the model
scores = model.evaluate(X_test, y_test, verbose=0)
print("Accuracy: {} \n Error: {}".format(scores[1], 100-scores[1]*100))

Epoch 1/10
59/59 - 37s - 634ms/step - accuracy: 0.6582 - loss: 1.2473 - val_accuracy: 0.8812 - val_loss: 0.4135
Epoch 2/10
59/59 - 34s - 582ms/step - accuracy: 0.9071 - loss: 0.3205 - val_accuracy: 0.9375 - val_loss: 0.2214
Epoch 3/10
59/59 - 34s - 576ms/step - accuracy: 0.9439 - loss: 0.1961 - val_accuracy: 0.9583 - val_loss: 0.1460
Epoch 4/10
59/59 - 34s - 579ms/step - accuracy: 0.9587 - loss: 0.1428 - val_accuracy: 0.9662 - val_loss: 0.1146
Epoch 5/10
59/59 - 34s - 582ms/step - accuracy: 0.9653 - loss: 0.1164 - val_accuracy: 0.9714 - val_loss: 0.0963
Epoch 6/10
59/59 - 35s - 595ms/step - accuracy: 0.9709 - loss: 0.0986 - val_accuracy: 0.9747 - val_loss: 0.0858
Epoch 7/10
59/59 - 34s - 571ms/step - accuracy: 0.9738 - loss: 0.0870 - val_accuracy: 0.9783 - val_loss: 0.0742
Epoch 8/10
59/59 - 34s - 571ms/step - accuracy: 0.9761 - loss: 0.0807 - val_accuracy: 0.9798 - val_loss: 0.0665
Epoch 9/10
59/59 - 34s - 572ms/step - accuracy: 0.9784 - loss: 0.0715 - val_accuracy: 0.9818 - val_loss:

Double-click <b>here</b> for the solution.

<!-- Your answer is below:
# build the model
model = convolutional_model()

# fit the model
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=10, batch_size=1024, verbose=2)

# evaluate the model
scores = model.evaluate(X_test, y_test, verbose=0)
print("Accuracy: {} \n Error: {}".format(scores[1], 100-scores[1]*100))
-->


<h3>Practice Exercise 2</h3>


Now, let's see how number of epochs  affect the time required and accuracy of the model training. 
For this, you can keep the batch_size=1024 and epochs=25 and check it's effect on accuracy


In [ ]:
# Write your answer here

# build the model
model = convolutional_model()

# fit the model
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=25, batch_size=1024, verbose=2)

# evaluate the model
scores = model.evaluate(X_test, y_test, verbose=0)
print("Accuracy: {} \n Error: {}".format(scores[1], 100-scores[1]*100))

Epoch 1/25
59/59 - 35s - 594ms/step - accuracy: 0.6249 - loss: 1.4059 - val_accuracy: 0.8800 - val_loss: 0.4288
Epoch 2/25
59/59 - 33s - 565ms/step - accuracy: 0.9138 - loss: 0.2972 - val_accuracy: 0.9458 - val_loss: 0.1961
Epoch 3/25
59/59 - 33s - 562ms/step - accuracy: 0.9477 - loss: 0.1798 - val_accuracy: 0.9591 - val_loss: 0.1442
Epoch 4/25
59/59 - 34s - 577ms/step - accuracy: 0.9587 - loss: 0.1407 - val_accuracy: 0.9673 - val_loss: 0.1164
Epoch 5/25
59/59 - 34s - 571ms/step - accuracy: 0.9650 - loss: 0.1167 - val_accuracy: 0.9719 - val_loss: 0.0982
Epoch 6/25
59/59 - 34s - 573ms/step - accuracy: 0.9700 - loss: 0.1009 - val_accuracy: 0.9741 - val_loss: 0.0897
Epoch 7/25
59/59 - 33s - 567ms/step - accuracy: 0.9732 - loss: 0.0890 - val_accuracy: 0.9771 - val_loss: 0.0785
Epoch 8/25
59/59 - 33s - 567ms/step - accuracy: 0.9762 - loss: 0.0793 - val_accuracy: 0.9786 - val_loss: 0.0729
Epoch 9/25
59/59 - 33s - 563ms/step - accuracy: 0.9772 - loss: 0.0723 - val_accuracy: 0.9797 - val_loss:

Double-click <b>here</b> for the solution.

<!-- Your answer is below:
# build the model
model = convolutional_model()

# fit the model
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=25, batch_size=1024, verbose=2)

# evaluate the model
scores = model.evaluate(X_test, y_test, verbose=0)
print("Accuracy: {} \n Error: {}".format(scores[1], 100-scores[1]*100))


    -->


### Thank you for completing this lab!

This notebook was created by [Alex Aklson](https://www.linkedin.com/in/aklson/). I hope you found this lab interesting and educational. Feel free to contact me if you have any questions!


<!--
## Change Log

|  Date (YYYY-MM-DD) |  Version | Changed By  |  Change Description |
|---|---|---|---|
| 2024-11-20  | 3.0  | Aman  |  Updated the library versions to current |
| 2020-09-21  | 2.0  | Srishti  |  Migrated Lab to Markdown and added to course repo in GitLab |



<hr>

## <h3 align="center"> © IBM Corporation. All rights reserved. <h3/>


## <h3 align="center"> &#169; IBM Corporation. All rights reserved. <h3/>

